# ChaoSSM — Phase 3b: confirmatory battery (the manuscript's main table)

## Role of this phase

Every gate has already fired. Phase 4 showed chaos confers no benefit; Phase 6/6b showed *why*
(the trained gate holds the recurrence in contraction). This phase is **confirmatory**: it
produces the single, adequately-powered, harness-fixed results table that the manuscript reports,
so no reviewer can say the headline numbers came from an under-powered pilot.

It is NOT a gate and NOT exploratory. It re-runs the settled comparison at manuscript grade:

- **Six arms:** GRU (working-model reference), linear (non-selective nonlinear floor), mamba2
  (diagonal selective-SSM baseline), chaossm_free, chaossm_corridor, and -- as the visible
  embodiment of the Phase 4 finding -- chaossm_free_periodic (the periodic-r twin).
- **Three tasks:** copy, selective copy, associative recall.
- **n = 15 seeds** (the manuscript-grade sample the gates ran below at n=5/8).
- **Fixed, documented lrs** from the Phase 3b'/4 sweeps -- NO tuning here, so the comparison is
  clean and the runtime is predictable.
- **Held-out eval** (fixed 2,000-sequence set per task), **effect sizes** (Cliff's delta),
  **bootstrap 95% CIs**, and **Holm-Bonferroni** correction across arm pairs per task.
- Optional **fine-tune fragility** block (FRAGILITY=True) closing the open N=192 question.
- Optional **scale** at N in {48,96,192} (SCALE=True).

## Runtime discipline (learned from earlier phases)

The corridor arm carries the lambda-gated jacrev tax; all other arms are cheap. Default scope is
one N (48), three tasks, six arms, 15 seeds -> 6*3*15 = 270 runs. Budget from the *measured*
Phase 4 timing (~6 min cheap / ~10 min corridor on a loaded T4), and the notebook prints a
projection before the loop plus a measured refresh after the first arm. If a single T4 session is
tight, run tasks one at a time (TASKS_SUBSET) or drop to n=10 for a first pass.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field
from itertools import combinations
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))
def setup_results_dir(project="ChaoSSM",phase="phase3b_battery"):
    try:
        from google.colab import drive; drive.mount("/content/drive",force_remount=False)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"./{project}_results/{phase}"); print(f"[info] not colab ({type(e).__name__})")
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
gpu: Tesla T4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase3b_battery


## Configuration

In [2]:
SMOKE_TEST=False
FRAGILITY=False          # add fine-tune fragility block (closes open N=192 question)
SCALE=False              # run N in {48,96,192} instead of just 48
TASKS_SUBSET=None        # e.g. ("copy",) to run one task per session; None = all three

@dataclass
class Config:
    state_dim:int=48; d_model:int=48; vocab:int=16; n_layers:int=2; mlp_mult:int=4
    steps:int=5000; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; gamma_init:float=1.0
    dual_lr:float=0.05; alpha_max:float=50.0; corridor_target:float=0.15; corridor_half_width:float=0.10
    lyap_window:int=4; lyap_subsample:int=3; lyap_every:int=20
    r_cap_chaotic:float=4.0; r_cap_periodic:float=3.4
    arms:tuple=("gru","linear","mamba2","chaossm_free","chaossm_free_periodic","chaossm_corridor")
    tasks:tuple=("copy","selective_copy","recall")
    seeds:int=15
    eval_n:int=2000; eval_seed:int=12345; eval_every:int=1000
    copy_len:int=12; sel_len:int=32; sel_keep:int=6; recall_pairs:int=8
    scale_N:tuple=(48,96,192)
    # fixed lrs from Phase 3b'/4 sweeps -- documented, not tuned here
    lr:dict=field(default_factory=lambda: dict(
        gru=3e-3, linear=1e-4, mamba2=3e-3,
        chaossm_free=1e-4, chaossm_free_periodic=1e-4, chaossm_corridor=1e-4))
    ft_steps:int=1000       # fine-tune duration for fragility block
    ft_lr:float=1e-4

CFG=Config()
if TASKS_SUBSET: CFG.tasks=tuple(TASKS_SUBSET)
if SMOKE_TEST:
    CFG.steps=40; CFG.seeds=2; CFG.eval_n=20; CFG.eval_every=20; CFG.ft_steps=10
    print("*** SMOKE TEST ***")
print(f"FRAGILITY={FRAGILITY} SCALE={SCALE} tasks={CFG.tasks} seeds={CFG.seeds}")
print(json.dumps({k:v for k,v in asdict(CFG).items()},indent=2,default=str))
def arm_is_periodic(a): return a.endswith("_periodic")
def arm_is_chaossm(a): return a.startswith("chaossm")
def arm_uses_corridor(a): return a.startswith("chaossm_corridor")
def arm_needs_traj(a): return arm_uses_corridor(a)
def is_lyap_step(cfg,s,w): return s>=w and s%cfg.lyap_every==0

FRAGILITY=False SCALE=False tasks=('copy', 'selective_copy', 'recall') seeds=15
{
  "state_dim": 48,
  "d_model": 48,
  "vocab": 16,
  "n_layers": 2,
  "mlp_mult": 4,
  "steps": 5000,
  "batch": 64,
  "warmup_frac": 0.05,
  "lr_min_frac": 0.1,
  "grad_clip": 1.0,
  "gamma_init": 1.0,
  "dual_lr": 0.05,
  "alpha_max": 50.0,
  "corridor_target": 0.15,
  "corridor_half_width": 0.1,
  "lyap_window": 4,
  "lyap_subsample": 3,
  "lyap_every": 20,
  "r_cap_chaotic": 4.0,
  "r_cap_periodic": 3.4,
  "arms": [
    "gru",
    "linear",
    "mamba2",
    "chaossm_free",
    "chaossm_free_periodic",
    "chaossm_corridor"
  ],
  "tasks": [
    "copy",
    "selective_copy",
    "recall"
  ],
  "seeds": 15,
  "eval_n": 2000,
  "eval_seed": 12345,
  "eval_every": 1000,
  "copy_len": 12,
  "sel_len": 32,
  "sel_keep": 6,
  "recall_pairs": 8,
  "scale_N": [
    48,
    96,
    192
  ],
  "lr": {
    "gru": 0.003,
    "linear": 0.0001,
    "mamba2": 0.003,
    "chaossm_free": 0.0001,
    "chaossm_free_pe

## 1. Cells + block + model (identical to Phase 4/6)

In [3]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)
class LinearMap(nn.Module):
    def __init__(self,N):
        super().__init__(); W=torch.empty(N,N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.5)
    def forward(self,h): return torch.tanh(h@self.W_lin.T)
class ChaoticMap(nn.Module):
    def __init__(self,N,r_cap,gi=1.0):
        super().__init__(); self.r_cap=float(r_cap)
        self.w_r=nn.Parameter(torch.zeros(N))
        W=torch.empty(N,N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(gi)))))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
class ChaoSSMCell(nn.Module):
    def __init__(self,cfg,arm):
        super().__init__(); self.N=cfg.state_dim; self.is_ch=arm_is_chaossm(arm)
        if self.is_ch:
            r_cap=cfg.r_cap_periodic if arm_is_periodic(arm) else cfg.r_cap_chaotic
            self.phi=ChaoticMap(cfg.state_dim,r_cap,cfg.gamma_init)
        else: self.phi=LinearMap(cfg.state_dim)
        self.A=nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9,1.1)))
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.W_g=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.b_g=nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self,h,x):
        g=torch.sigmoid(x@self.W_g.T+self.b_g)
        raw=g*self.phi(h@self.A.T)+(1-g)*h+x@self.B.T
        return triangle_fold(raw) if self.is_ch else torch.tanh(raw)
    def init_state(self,b,dev,dt):
        v=0.5 if self.is_ch else 0.0; return torch.full((b,self.N),v,device=dev,dtype=dt)
class Mamba2Cell(nn.Module):
    def __init__(self,cfg):
        super().__init__(); self.N=cfg.state_dim
        self.A_log=nn.Parameter(torch.log(torch.arange(1,cfg.state_dim+1,dtype=torch.float32)))
        self.W_delta=nn.Linear(cfg.state_dim,cfg.state_dim); self.W_B=nn.Linear(cfg.state_dim,cfg.state_dim)
        nn.init.zeros_(self.W_delta.bias); nn.init.zeros_(self.W_B.bias)
    def step(self,h,x):
        delta=F.softplus(self.W_delta(x)); A=-torch.exp(self.A_log)
        return torch.exp(delta*A)*h+delta*self.W_B(x)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
class GRUArmCell(nn.Module):
    def __init__(self,cfg): super().__init__(); self.N=cfg.state_dim; self.cell=nn.GRUCell(cfg.state_dim,cfg.state_dim)
    def step(self,h,x): return self.cell(x,h)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
def make_cell(cfg,arm):
    if arm=="mamba2": return Mamba2Cell(cfg)
    if arm=="gru": return GRUArmCell(cfg)
    return ChaoSSMCell(cfg,arm)
class Block(nn.Module):
    def __init__(self,cfg,arm):
        super().__init__(); self.arm=arm
        self.ln1=nn.LayerNorm(cfg.d_model); self.cell=make_cell(cfg,arm)
        self.ln2=nn.LayerNorm(cfg.d_model); h=cfg.d_model*cfg.mlp_mult
        self.mlp=nn.Sequential(nn.Linear(cfg.d_model,h),nn.GELU(),nn.Linear(h,cfg.d_model))
    def step(self,hs,x):
        u=self.ln1(x); hs=self.cell.step(hs,u); y=x+hs; y=y+self.mlp(self.ln2(y)); return hs,y
    def init_state(self,b,dev,dt): return self.cell.init_state(b,dev,dt)
class ChaoSSMLM(nn.Module):
    def __init__(self,cfg,arm,V,R):
        super().__init__(); self.cfg=cfg; self.arm=arm
        self.emb=nn.Embedding(V,cfg.d_model)
        self.blocks=nn.ModuleList([Block(cfg,arm) for _ in range(cfg.n_layers)])
        self.ln_f=nn.LayerNorm(cfg.d_model); self.readout=nn.Linear(cfg.d_model,R)
    def forward(self,tok,return_traj=False):
        B,T=tok.shape; x=self.emb(tok); dt=x.dtype
        st=[blk.init_state(B,tok.device,dt) for blk in self.blocks]; outs=[]
        trajs=[[] for _ in self.blocks] if return_traj else None
        xseqs=[[] for _ in self.blocks] if return_traj else None
        for t in range(T):
            xt=x[:,t]
            for li,blk in enumerate(self.blocks):
                hi=xt; st[li],xt=blk.step(st[li],xt)
                if return_traj and arm_needs_traj(blk.arm): trajs[li].append(st[li]); xseqs[li].append(blk.ln1(hi))
            outs.append(self.readout(self.ln_f(xt)))
        lg=torch.stack(outs,1)
        if not return_traj: return lg
        to={li:(torch.stack(trajs[li],1),torch.stack(xseqs[li],1)) for li in range(len(self.blocks)) if trajs[li]}
        return lg,to

## 2. Tasks + held-out eval (copy / selective copy / recall)

In [4]:
V=CFG.vocab; SEP,PAD,QCUE=V,V+1,V+2; VOCAB_SIZE=V+3; READOUT=V+3; CHANCE=1.0/V
def make_copy(cfg,batch,device,gen):
    L=cfg.copy_len; src=torch.randint(0,V,(batch,L),device=device,generator=gen)
    inp=torch.cat([src,torch.full((batch,1),SEP,device=device),torch.full((batch,L),PAD,device=device)],1)
    tgt=torch.cat([torch.full((batch,L+1),-100,device=device),src],1); return inp,tgt
def make_selcopy(cfg,batch,device,gen):
    Ls,K=cfg.sel_len,cfg.sel_keep
    stream=torch.randint(0,V,(batch,Ls),device=device,generator=gen)
    keep=torch.stack([torch.sort(torch.randperm(Ls,generator=gen,device=device)[:K]).values for _ in range(batch)])
    ks=[set(k.tolist()) for k in keep]; ri,rt=[],[]
    for b in range(batch):
        seq=[]
        for i in range(Ls):
            if i in ks[b]: seq.append(QCUE)
            seq.append(int(stream[b,i]))
        kv=[int(stream[b,i]) for i in sorted(ks[b])]
        ri.append(seq+[SEP]+[PAD]*K); rt.append([-100]*(len(seq)+1)+kv)
    m=max(len(r) for r in ri)
    for r in ri: r+=[PAD]*(m-len(r))
    for r in rt: r+=[-100]*(m-len(r))
    return torch.tensor(ri,device=device),torch.tensor(rt,device=device)
def make_recall(cfg,batch,device,gen):
    P=cfg.recall_pairs
    keys=torch.stack([torch.randperm(V,generator=gen,device=device)[:P] for _ in range(batch)])
    vals=torch.randint(0,V,(batch,P),device=device,generator=gen)
    qp=torch.randint(0,P,(batch,),device=device,generator=gen); ri,rt=[],[]
    for b in range(batch):
        seq=[]
        for i in range(P): seq+=[int(keys[b,i]),int(vals[b,i])]
        qk=int(keys[b,qp[b]]); ans=int(vals[b,qp[b]])
        ri.append(seq+[QCUE,qk,SEP,PAD]); rt.append([-100]*(len(seq)+3)+[ans])
    return torch.tensor(ri,device=device),torch.tensor(rt,device=device)
TASK_FN=dict(copy=make_copy,selective_copy=make_selcopy,recall=make_recall)
def make_batch(task,cfg,batch,device,gen): return TASK_FN[task](cfg,batch,device,gen)
def build_eval(cfg):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    return {t:make_batch(t,cfg,cfg.eval_n,DEVICE,g) for t in cfg.tasks}
EVAL={}
@torch.no_grad()
def eval_ho(model,task,chunk=256):
    model.eval(); ia,ta=EVAL[task]; c=tot=0.0
    for i in range(0,ia.shape[0],chunk):
        inp=ia[i:i+chunk]; tg=ta[i:i+chunk]; lg=model(inp); m=tg!=-100; p=lg.argmax(-1)
        c+=((p==tg)&m).float().sum().item(); tot+=m.float().sum().item()
    model.train(); return c/max(tot,1)

## 3. proxy lambda + trainer (lambda-gated; supports fine-tune)

In [5]:
from torch.func import jacrev
def _sv(cell,h,x): return cell.step(h.unsqueeze(0),x.unsqueeze(0)).squeeze(0)
def lambda_proxy(cell,traj,xseq,W,ns):
    B,T,N=traj.shape
    if T<=W+1: return torch.zeros((),device=traj.device)
    ls=[]
    for b in torch.randperm(B)[:ns]:
        t0=int(torch.randint(0,T-W-1,(1,))); M=torch.eye(N,device=traj.device,dtype=traj.dtype)
        for k in range(W): M=jacrev(lambda z:_sv(cell,z,xseq[b,t0+k]))(traj[b,t0+k])@M
        ls.append(torch.log(torch.linalg.matrix_norm(M,ord=2)+1e-12)/W)
    return torch.stack(ls).mean()
def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn
def train_arm(arm,task,seed,cfg,return_model=False):
    torch.manual_seed(seed); np.random.seed(seed)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    model=ChaoSSMLM(cfg,arm,VOCAB_SIZE,READOUT).to(DEVICE)
    lr=cfg.lr[arm]; opt=torch.optim.AdamW(model.parameters(),lr=lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(cfg,cfg.steps))
    use=arm_uses_corridor(arm); ct=cfg.corridor_target
    lo,hi=ct-cfg.corridor_half_width,ct+cfg.corridor_half_width; a_hi=a_lo=0.0
    warm=max(1,int(cfg.warmup_frac*cfg.steps)); div=False
    for step in range(cfg.steps):
        inp,tgt=make_batch(task,cfg,cfg.batch,DEVICE,gen)
        need=use and is_lyap_step(cfg,step,warm)
        out=model(inp,return_traj=need); lg,traj=out if need else (out,None)
        Lt=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1),ignore_index=-100)
        Lp=torch.zeros((),device=DEVICE); lam=float("nan")
        if need and traj:
            ls=[]
            for li,(tr,xs) in traj.items():
                lv=lambda_proxy(model.blocks[li].cell,tr,xs,cfg.lyap_window,cfg.lyap_subsample); ls.append(lv)
                Lp=Lp+a_hi*F.relu(lv-hi)+a_lo*F.relu(lo-lv)
            if ls: lam=float(torch.stack(ls).mean().detach())
        opt.zero_grad(set_to_none=True); (Lt+Lp).backward()
        gn=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip).item(); opt.step(); sched.step()
        if use and math.isfinite(lam):
            a_hi=float(np.clip(a_hi+cfg.dual_lr*(lam-hi),0,cfg.alpha_max)); a_lo=float(np.clip(a_lo+cfg.dual_lr*(lo-lam),0,cfg.alpha_max))
        if not (math.isfinite(Lt.item()) and math.isfinite(gn)): div=True; break
    acc=eval_ho(model,task) if not div else float("nan")
    rec=dict(arm=arm,task=task,seed=seed,N=cfg.state_dim,lr=lr,diverged=div,held_out_acc=acc)
    return (model,rec) if return_model else rec

## 4. Stats helpers

In [6]:
def cliffs_delta(x,y):
    x=np.asarray(x); y=np.asarray(y)
    gt=sum((xi>y).sum() for xi in x); lt=sum((xi<y).sum() for xi in x)
    return (gt-lt)/(len(x)*len(y))
def boot_ci(x,y,n=10000,seed=0):
    rng=np.random.default_rng(seed); x=np.asarray(x); y=np.asarray(y); d=np.empty(n)
    for i in range(n): d[i]=rng.choice(x,len(x),True).mean()-rng.choice(y,len(y),True).mean()
    return float(np.percentile(d,2.5)),float(np.percentile(d,97.5))
def holm(pv):
    idx=np.argsort(pv); m=len(pv); adj=np.empty(m); run=0.0
    for rank,i in enumerate(idx):
        run=max(run,(m-rank)*pv[i]); adj[i]=min(run,1.0)
    return adj

## 5. Main battery -- N in scale set x tasks x arms x seeds

In [ ]:
def project_runtime(cfg,Ns):
    per=lambda a:(10.0 if arm_uses_corridor(a) else 5.0)*(cfg.steps/5000.0)
    est=sum(per(a) for a in cfg.arms)*len(cfg.tasks)*cfg.seeds*len(Ns)
    n=len(cfg.arms)*len(cfg.tasks)*cfg.seeds*len(Ns)
    print(f"runs: {n}  |  prior estimate (T4): ~{est/60:.1f} h  [{est:.0f} min]  -- refine after arm 1")
    return est

Ns=list(CFG.scale_N) if SCALE else [CFG.state_dim]
project_runtime(CFG,Ns)
rows=[]; t0=time.time(); first=None; total=len(CFG.arms)*len(CFG.tasks)*CFG.seeds*len(Ns); done=0
for N in Ns:
    cfgN=Config(**{**asdict(CFG)}); cfgN.state_dim=N; cfgN.d_model=N
    cfgN.tasks=CFG.tasks; cfgN.lr=CFG.lr
    globals()["EVAL"]=build_eval(cfgN)
    for arm in cfgN.arms:
        at0=time.time()
        for task in cfgN.tasks:
            for sd in range(cfgN.seeds):
                r=train_arm(arm,task,sd,cfgN); rows.append(r); done+=1
                fl=" DIV" if r["diverged"] else ""
                print(f"[{done}/{total}] N={N} {arm:22s} {task:15s} s={sd:2d} acc={r['held_out_acc']:.4f}{fl}")
        if first is None:
            first=time.time()-at0; per=first/(len(cfgN.tasks)*cfgN.seeds)
            print(f"    [timing] first arm {first/60:.1f} min (~{per:.0f}s/run); "
                  f"projected total ~{per*total/60:.1f} min")
RES=pd.DataFrame(rows); savetable(RES,"phase3b_battery_summary.csv")
print(f"\nwall clock: {(time.time()-t0)/60:.1f} min")
print(RES.groupby(["N","arm","task"]).held_out_acc.mean().round(4).to_string())

runs: 270  |  prior estimate (T4): ~26.2 h  [1575 min]  -- refine after arm 1
[1/270] N=48 gru                    copy            s= 0 acc=0.9926
[2/270] N=48 gru                    copy            s= 1 acc=0.9908
[3/270] N=48 gru                    copy            s= 2 acc=0.9927


## 6. Manuscript table + pairwise stats (per task, at base N)

In [ ]:
baseN=CFG.state_dim
sub=RES[RES.N==baseN]
# summary table: mean +/- std per arm x task
piv=sub.groupby(["task","arm"]).held_out_acc.agg(["mean","std","count"]).round(4).reset_index()
savetable(piv,"phase3b_battery_table.csv")
print(piv.to_string(index=False))

stat_rows=[]
for task in CFG.tasks:
    arms=list(CFG.arms)
    pvals=[]; keys=[]
    for a,b_ in combinations(arms,2):
        x=sub[(sub.arm==a)&(sub.task==task)].sort_values("seed").held_out_acc.values
        y=sub[(sub.arm==b_)&(sub.task==task)].sort_values("seed").held_out_acc.values
        x=x[np.isfinite(x)]; y=y[np.isfinite(y)]; n=min(len(x),len(y))
        if n<3 or np.allclose(x[:n],y[:n]): p=np.nan
        else:
            try: _,p=wilcoxon(x[:n],y[:n])
            except ValueError: p=np.nan
        dlt=cliffs_delta(x,y) if len(x) and len(y) else np.nan
        lo,hi=boot_ci(x,y) if len(x) and len(y) else (np.nan,np.nan)
        stat_rows.append(dict(task=task,pair=f"{a} vs {b_}",n=n,p_raw=p,
                              cliffs_delta=round(dlt,3),mean_diff=round(float(np.mean(x)-np.mean(y)),4),
                              ci_low=round(lo,4),ci_high=round(hi,4)))
        pvals.append(p); keys.append((task,f"{a} vs {b_}"))
    # holm within task on the finite pvals
    fin=[i for i,p in enumerate(pvals) if p==p]
    if fin:
        adj=holm(np.array([pvals[i] for i in fin]))
        for j,i in enumerate(fin):
            for sr in stat_rows:
                if sr["task"]==keys[i][0] and sr["pair"]==keys[i][1]: sr["p_holm"]=round(float(adj[j]),4)
STATS=pd.DataFrame(stat_rows); savetable(STATS,"phase3b_battery_stats.csv")
print("\nKey contrasts (vs GRU and periodic-vs-chaotic):")
show=STATS[STATS.pair.str.contains("gru")|STATS.pair.str.contains("free vs chaossm_free_periodic")]
print(show.to_string(index=False))

## 7. Figure -- manuscript main bars

In [ ]:
colors={"gru":"tab:purple","linear":"gray","mamba2":"tab:orange",
        "chaossm_free":"tab:blue","chaossm_free_periodic":"tab:cyan","chaossm_corridor":"tab:green"}
fig,axes=plt.subplots(1,len(CFG.tasks),figsize=(5*len(CFG.tasks),4.3),sharey=True)
if len(CFG.tasks)==1: axes=[axes]
for ax,task in zip(axes,CFG.tasks):
    order=list(CFG.arms)
    ms=[sub[(sub.arm==a)&(sub.task==task)].held_out_acc.mean() for a in order]
    ss=[sub[(sub.arm==a)&(sub.task==task)].held_out_acc.std() for a in order]
    ax.bar(range(len(order)),ms,yerr=ss,capsize=3,color=[colors[a] for a in order])
    ax.axhline(CHANCE,color="r",ls="--",lw=1)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([a.replace("chaossm_","").replace("_periodic","·per") for a in order],rotation=40,ha="right",fontsize=8)
    ax.set_title(task); ax.set_ylim(0,1.02); ax.grid(alpha=.3,axis="y")
axes[0].set_ylabel(f"held-out accuracy (n={CFG.seeds})")
fig.suptitle(f"Phase 3b: confirmatory battery, N={baseN}")
fig.tight_layout(); savefig(fig,"phase3b_battery_bars.png"); plt.close(fig)

## 8. Fine-tune fragility (FRAGILITY=True) -- closes the N=192 question

In [ ]:
FRAG=None
if FRAGILITY:
    frag_rows=[]
    fN=list(CFG.scale_N) if SCALE else [CFG.state_dim]
    for N in fN:
        cfgN=Config(**asdict(CFG)); cfgN.state_dim=N; cfgN.d_model=N; cfgN.tasks=("copy",); cfgN.lr=CFG.lr
        globals()["EVAL"]=build_eval(cfgN)
        for arm in cfgN.arms:
            for sd in range(min(cfgN.seeds,8)):
                model,rec=train_arm(arm,"copy",sd,cfgN,return_model=True)
                if rec["diverged"]: continue
                acc0=rec["held_out_acc"]
                # fine-tune on selective_copy briefly, then re-measure copy (catastrophic drift proxy)
                gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(1000+sd)
                opt=torch.optim.AdamW(model.parameters(),lr=cfgN.ft_lr)
                for _ in range(cfgN.ft_steps):
                    inp,tgt=make_batch("selective_copy",cfgN,cfgN.batch,DEVICE,gen)
                    lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1),ignore_index=-100)
                    opt.zero_grad(set_to_none=True); L.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(),cfgN.grad_clip); opt.step()
                acc1=eval_ho(model,"copy")
                frag_rows.append(dict(N=N,arm=arm,seed=sd,acc_before=acc0,acc_after=acc1,acc_drop=acc0-acc1))
                print(f"N={N} {arm:22s} s={sd} copy {acc0:.3f}->{acc1:.3f} drop {acc0-acc1:+.3f}")
    FRAG=pd.DataFrame(frag_rows); savetable(FRAG,"phase3b_fragility.csv")
    print(FRAG.groupby(["N","arm"])["acc_drop"].mean().round(4).to_string())
else:
    print("FRAGILITY=False -> skipped.")

## 9. Verdict

In [ ]:
def m(task,arm):
    s=sub[(sub.arm==arm)&(sub.task==task)].held_out_acc
    return round(float(s.mean()),4) if len(s) else None
verdict=dict(
    smoke_test=SMOKE_TEST, base_N=baseN, seeds=CFG.seeds, tasks=list(CFG.tasks), arms=list(CFG.arms),
    means={t:{a:m(t,a) for a in CFG.arms} for t in CFG.tasks},
    scale=SCALE, fragility_ran=bool(FRAG is not None),
    any_diverged=bool(RES.diverged.any()),
    note=("Confirmatory battery. Expected pattern (from gates): GRU highest on all tasks; chaossm "
          "arms well below GRU; chaossm_free ~ chaossm_free_periodic (Phase 4 periodic-r result "
          "reproduced at n=15); mamba2 intermediate. This table is the manuscript's main result; "
          "the mechanism is Phase 6b. Effect sizes + Holm-corrected p-values in phase3b_battery_stats.csv."),
)
savejson(verdict,"phase3b_battery_verdict.json")
print(json.dumps(verdict,indent=2,default=str))